This MATLAB script processes TomoSAR (Tomographic Synthetic Aperture Radar) data. It begins by setting various options related to the data directory, radar band, heading, acquisition mode, and polarimetric channel. It then determines indices and strings based on the acquisition mode and polarimetric channel.

The script loads local coordinates and DTM MetaSensing data from specified file paths, including azimuth and ground range axes, as well as the digital elevation model (DEM) image. It constructs a data cube (Pcube) and a normalized data cube (PcubeNorm) by loading data for each z level from corresponding data files.

The script visualizes the normalized power data using imagesc. It also loads additional data for off-nadir angle, ground slope, and azimuth slope from specified data files.

Next, the script sets correction parameters based on the radar band, heading, and acquisition mode. It computes a correction map in decibels (dB) using these parameters and the depression angle data, and applies the correction to the averaged canopy power data to obtain Pcanopy_de_ramped_dB.

Finally, the script visualizes the canopy power data before and after applying the correction, and loads latitude and longitude images from the MetaSensing data file for geocoding the data cube. This provides a complete processing pipeline for loading, correcting, and visualizing TomoSAR data.

#  Definitions 

### Ground Range Axis

**Concept:**

- **Ground Range**: The ground range axis represents the distance along the ground from the radar to the target. It is the projection of the slant range (the direct line distance from the radar to the target) onto the ground plane.
- **Data Representation**: This data typically consists of a series of distance values, each corresponding to a specific location on the ground. These values are often in meters or kilometers.
- **Purpose**: The ground range axis is used to map the radar data to specific ground locations. It helps in understanding where the targets or features detected by the radar are located on the Earth's surface.

**Example:**

Imagine a radar on an aircraft flying over a landscape. The ground range axis data would tell you how far away each point on the ground is from the radar's position, measured along the ground.

### Azimuth Axis

**Concept:**

- **Azimuth**: The azimuth axis represents the direction along the flight path of the radar platform. In SAR imaging, this direction is parallel to the movement of the radar platform (e.g., an aircraft or satellite).
- **Data Representation**: This data consists of a series of time values or indices corresponding to the radar's position along its flight path. Each value represents a different point in time or position along the flight path.
- **Purpose**: The azimuth axis is used to organize the radar data according to the radar platform's movement. It helps in constructing the 2D image by combining data collected over time as the radar platform moves.


### Cross Range

**Concept:**

- **Cross Range**: The cross range axis refers to the dimension perpendicular to both the ground range and the direction of the radar platform's motion. In Synthetic Aperture Radar (SAR) imaging, it represents the angular distance along the radar's line of sight. Cross range is essential for achieving high resolution in the direction orthogonal to the radar's flight path.

**Data Representation:**

- **Data Type**: The data along the cross range axis typically involves spatial frequencies or angular positions. These values are used to reconstruct the image in the cross range dimension.
- **Resolution**: The resolution in the cross range direction is determined by the synthetic aperture created by the radar's movement over time, rather than the physical aperture size. This synthetic aperture is formed by combining multiple radar returns taken from different positions along the flight path.

**Purpose:**

- **Image Formation**: Cross range data is crucial for constructing high-resolution radar images. By processing the data from multiple radar returns, the SAR system can create a detailed image with fine resolution in both the range and cross range dimensions.
- **Enhanced Detail**: The cross range resolution enables the identification and discrimination of features that are close together in the horizontal plane, perpendicular to the radar's motion.

**Example:**

Consider a SAR system on a satellite orbiting the Earth. As the satellite moves along its orbit, it collects radar signals from various angles relative to the ground. The cross range axis data allows the SAR system to combine these signals and produce a detailed image with high resolution across the swath of the radar's coverage area.



#  Run script 

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from netCDF4 import Dataset
# from read_biomass_data import read_biomass_data

In [3]:
import numpy as np

def read_biomass_data(full_path, crop_struct, data_type):
    """
    Reads files produced by write_biomass_data.m.

    Parameters:
    full_path (str): Path to the file to be read.
    crop_struct (dict): Dictionary with the following keys:
        S: number of samples (dia only)
        L: number of lines (dia only)
        Sin: first sample to be read (starting from 1)
        Sfin: last sample to be read
        Lin: first line to be read (starting from 1)
        Lfin: last line to be read
    data_type (int): Type of data to read (1 for Complex, 2 for Real).

    Returns:
    np.ndarray: The read data as a NumPy array.
    """
    # Single
    N = 4

    try:
        fid = open(full_path, 'rb')  # Open the file in binary read mode
    except IOError:
        print('Error in read_biomass_data: File cannot be opened.')
        return None

    # Read the total number of rows and columns
    NrTot = np.fromfile(fid, dtype=np.float32, count=1)[0]
    NcTot = np.fromfile(fid, dtype=np.float32, count=1)[0]

    # If crop_struct is not provided, use the entire data
    if not crop_struct:
        crop_struct = {
            'S': NrTot,
            'Sin': 1,
            'Sfin': NrTot,
            'L': NcTot,
            'Lin': 1,
            'Lfin': NcTot
        }

    # Check for mismatches between provided crop_struct and actual dimensions
    if 'S' in crop_struct and 'L' in crop_struct:
        if NrTot != crop_struct['S'] or NcTot != crop_struct['L']:
            print('Warning in read_biomass_data:')
            print(f'Nr mismatch: {NrTot} vs {crop_struct["S"]}')
            print(f'Nc mismatch: {NcTot} vs {crop_struct["L"]}')

    # Calculate the number of rows and columns to read
    Nr = crop_struct['Sfin'] - crop_struct['Sin'] + 1
    Nc = crop_struct['Lfin'] - crop_struct['Lin'] + 1

    # Read complex data
    if data_type == 1:
        # Move file pointer to the start of the relevant data block
        fid.seek(int(2 * N + ((crop_struct['Lin'] - 1) * NrTot + crop_struct['Sin'] - 1) * N * 2), 0)
        
        if crop_struct['Sin'] == 1 and crop_struct['Sfin'] == NrTot:
            # Read the entire block of complex data
            I = np.fromfile(fid, dtype=np.float32, count=2 * Nr * Nc).reshape(2 * Nr, Nc)
        else:
            # Read the cropped block of complex data
            bytes_to_skip = int((NrTot - crop_struct['Sfin'] + crop_struct['Sin'] - 1) * N * 2)
            I = np.zeros((2 * Nr, Nc), dtype=np.float32)
            for i in range(Nc):
                I[:, i] = np.fromfile(fid, dtype=np.float32, count=2 * Nr)
                fid.seek(bytes_to_skip, 1)
        
        # Combine real and imaginary parts into complex numbers
        I = I[0::2, :] + 1j * I[1::2, :]
        fid.close()
    
    # Read real data
    elif data_type == 2:
        # Move file pointer to the start of the relevant data block
        fid.seek(int(2 * N + ((crop_struct['Lin'] - 1) * NrTot + crop_struct['Sin'] - 1) * N), 0)
        
        if crop_struct['Sin'] == 1 and crop_struct['Sfin'] == NrTot:
            # Read the entire block of real data
            I = np.fromfile(fid, dtype=np.float32, count=Nr * Nc).reshape(Nr, Nc)
        else:
            # Read the cropped block of real data
            bytes_to_skip = int((NrTot - crop_struct['Sfin'] + crop_struct['Sin'] - 1) * N)
            I = np.zeros((Nr, Nc), dtype=np.float32)
            for i in range(Nc):
                I[:, i] = np.fromfile(fid, dtype=np.float32, count=Nr)
                fid.seek(bytes_to_skip, 1)
        
        fid.close()
    
    else:
        print('Error in read_biomass_data: Unrecognized data type.')
        return None

    return I


"""
Explanation:
File Handling: The file is opened in binary read mode ('rb'), and a check is added to handle file opening errors.
Reading Dimensions: The total number of rows and columns (NrTot and NcTot) are read from the file.
Default Crop Struct: If crop_struct is not provided, the function sets it to read the entire data.
Dimension Mismatch Warning: A warning is printed if there is a mismatch between the provided crop_struct and the actual dimensions in the file.
Data Reading:
For complex data (data_type == 1), the function reads pairs of real and imaginary parts and combines them.
For real data (data_type == 2), the function reads the real values directly.
Seek and Skip: The function uses seek to position the file pointer correctly and skips bytes as needed to read the cropped data correctly.
Error Handling: Basic error handling is added for unrecognized data types.
"""

"\nExplanation:\nFile Handling: The file is opened in binary read mode ('rb'), and a check is added to handle file opening errors.\nReading Dimensions: The total number of rows and columns (NrTot and NcTot) are read from the file.\nDefault Crop Struct: If crop_struct is not provided, the function sets it to read the entire data.\nDimension Mismatch Warning: A warning is printed if there is a mismatch between the provided crop_struct and the actual dimensions in the file.\nData Reading:\nFor complex data (data_type == 1), the function reads pairs of real and imaginary parts and combines them.\nFor real data (data_type == 2), the function reads the real values directly.\nSeek and Skip: The function uses seek to position the file pointer correctly and skips bytes as needed to read the cropped data correctly.\nError Handling: Basic error handling is added for unrecognized data types.\n"

In [ ]:

# Options
parent_path = 'D:/Dropbox/Dropbox (DEIB)/TomoSense/SAR_data_Matlab/'

# Either 'P' or 'L'
band_opt = 'P'

# Either 'NW' (290°) or 'SE' (110°)
heading = 'NW'

# Either 'Monostatic' or 'Bistatic' (available for L-band only)
acquisition_mode = 'Monostatic'

# 'HH', 'HV' or 'VV'
polarimetric_channel = 'HH'

# Determine the acquisition mode index and string
if acquisition_mode == 'Monostatic':
    mono_bi_ind = 1
    mono_bi_str = 'mono'
elif acquisition_mode == 'Bistatic':
    mono_bi_ind = 2
    mono_bi_str = 'bi'

# Determine the polarimetric channel index
if polarimetric_channel == 'HH':
    pol_ind = 1
elif polarimetric_channel == 'HV':
    pol_ind = 2
elif polarimetric_channel == 'VV':
    pol_ind = 3

# Load local coordinates for ground range and azimuth axis
ground_range_axis = loadmat(os.path.join(parent_path, band_opt, heading, 'Util', 'datacube', 'ground_range_axis.mat'))['ground_range_axis'].squeeze()
azimuth_axis = loadmat(os.path.join(parent_path, band_opt, heading, 'Util', 'datacube', 'azimuth_axis.mat'))['azimuth_axis'].squeeze()
Ngr = len(ground_range_axis)  # Number of ground range points
Naz = len(azimuth_axis)       # Number of azimuth points

# Determine the MetaSensing SLC full path based on band and heading options
if band_opt == 'P':
    if heading == 'NW':
        MetaSensing_slc_fullpath = os.path.join(parent_path, band_opt, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200722092915_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc')
    elif heading == 'SE':
        MetaSensing_slc_fullpath = os.path.join(parent_path, band_opt, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200723082647_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc')
elif band_opt == 'L':
    if heading == 'NW':
        MetaSensing_slc_fullpath = os.path.join(parent_path, band_opt, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200914160410_2_1.4G_HH_41_pres_16_fdc_0.sar.sig.nc')
    elif heading == 'SE':
        MetaSensing_slc_fullpath = os.path.join(parent_path, band_opt, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200914150909_2_1.4G_HH_41_pres_16_fdc_0.sar.sig.nc')

# Read the MetaSensing SLC data using netCDF4
with Dataset(MetaSensing_slc_fullpath, 'r') as ds:
    MetaSensing_slc_azimuth_axis = ds.variables['CrossRange'][:]
    MetaSensing_slc_ground_range_axis = ds.variables['GroundRange'][:]
    DTM = ds.variables['DEMImage'][:]

# Find the indices for azimuth and ground range based on the loaded axes
az_start_ind = np.argmin(np.abs(MetaSensing_slc_azimuth_axis - azimuth_axis[0]))
az_stop_ind = np.argmin(np.abs(MetaSensing_slc_azimuth_axis - azimuth_axis[-1]))
gr_start_ind = np.argmin(np.abs(MetaSensing_slc_ground_range_axis - ground_range_axis[0]))
gr_stop_ind = np.argmin(np.abs(MetaSensing_slc_ground_range_axis - ground_range_axis[-1]))

# Slice the DTM to match the indices found
DTM = DTM[gr_start_ind:gr_stop_ind+1, az_start_ind:az_stop_ind+1]

# Define the z-axis range and initialize the data cubes
z_axis = np.arange(-10, 62, 2)
Nz = len(z_axis)  # Number of z-axis points
Pcube = np.zeros((Ngr, Naz, Nz))  # Initialize power cube
PcubeNorm = np.zeros((Ngr, Naz, Nz))  # Initialize normalized power cube (tomographic coherence)

# # Function to read biomass data (assumed to be implemented)
# def read_biomass_data(filepath, param, mode):
#     # Placeholder function
#     # You need to implement this based on the actual data format
#     pass

# Loop through each z-axis point and load the corresponding data
for z_ind in range(Nz):
    print(f'Loading {z_axis[z_ind]}m ({z_ind+1}/{Nz})...')
    
    # Determine the file paths for the current z-slice
    if band_opt == 'P':
        curr_fullpath = os.path.join(parent_path, band_opt, heading, 'DataCube', 'ComplexSlice', f'tomo_slice_{z_axis[z_ind]}m_{polarimetric_channel}.dat')
        curr_fullpathP = os.path.join(parent_path, band_opt, heading, 'DataCube', 'Power', f'Pbp_{z_axis[z_ind]}m_{polarimetric_channel}.dat')
    elif band_opt == 'L':
        curr_fullpath = os.path.join(parent_path, band_opt, heading, 'DataCube', acquisition_mode, 'ComplexSlice', f'tomo_slice_{z_axis[z_ind]}m_{polarimetric_channel}_{mono_bi_str}.dat')
        curr_fullpathP = os.path.join(parent_path, band_opt, heading, 'DataCube', acquisition_mode, 'Power', f'Pbp_{z_axis[z_ind]}m_{polarimetric_channel}_{mono_bi_str}.dat')

    # Load the data and normalize it
    Pcube[:, :, z_ind] = np.abs(read_biomass_data(curr_fullpath, [], 1))**2
    temp = read_biomass_data(curr_fullpathP, [], 2)
    PcubeNorm[:, :, z_ind] = Pcube[:, :, z_ind] / (temp + np.finfo(float).eps)  # Normalize to avoid division by zero
    print('done.')

# Plot the normalized power cube for a specific ground range slice
plt.figure()
plt.imshow(np.transpose(PcubeNorm[Ngr//3, :, :], (1, 0)), aspect='auto', extent=[azimuth_axis[0]*1e-3, azimuth_axis[-1]*1e-3, z_axis[-1], z_axis[0]])
plt.axis('xy')
plt.xlabel('Azimuth [km]')
plt.ylabel('[m]')
plt.show()


In [ ]:


# Loading angles
curr_fullpath = os.path.join(parent_path, band_opt, heading, 'Util', 'datacube', 'off_nadir_angle.dat')
depression_angle = read_biomass_data(curr_fullpath, [], 2)  # Read the off-nadir angle data

curr_fullpath = os.path.join(parent_path, band_opt, heading, 'Util', 'datacube', 'ground_slope.dat')
ground_slope = read_biomass_data(curr_fullpath, [], 2)  # Read the ground slope data

curr_fullpath = os.path.join(parent_path, band_opt, heading, 'Util', 'datacube', 'azimuth_slope.dat')
azimuth_slope = read_biomass_data(curr_fullpath, [], 2)  # Read the azimuth slope data

# Applying near range ramp correction

# Correction map parameters based on band option, heading, and acquisition mode
if band_opt == 'L':
    if heading == 'SE':
        if acquisition_mode == 'Bistatic':
            corr_param = np.array([[-0.018941605192216, -0.021898940712299, -0.021679719047565],
                                   [1.702595647297680, 1.931565689532339, 1.913967459859081],
                                   [-33.857334799222450, -43.841825739326303, -40.862641121416431]])
        elif acquisition_mode == 'Monostatic':
            corr_param = np.array([[-0.014765665681876, -0.017005078160286, -0.014740433884030],
                                   [1.330766957008727, 1.482310610364879, 1.352307685660809],
                                   [-24.995145750297052, -33.153291354242981, -28.035010608712081]])
    elif heading == 'NW':
        if acquisition_mode == 'Bistatic':
            corr_param = np.array([[-0.012365319880992, -0.012448077495719, -0.011384198035345],
                                   [1.262527846970460, 1.262170743310784, 1.215788898014319],
                                   [-34.820607853988491, -41.840714971075407, -36.659242369319529]])
        elif acquisition_mode == 'Monostatic':
            corr_param = np.array([[-0.013860196998189, -0.016160008067196, -0.013312410070442],
                                   [1.304311362829773, 1.566269858245916, 1.333795748421465],
                                   [-32.517910903712298, -43.874405580764019, -35.995456119269875]])
    depr_angle_max = 42.5  # Maximum depression angle for L-band
elif band_opt == 'P':
    if heading == 'SE':
        corr_param = np.array([[-0.007933997591323, -0.004547124432006, -0.003496405945491],
                               [0.935552134822399, 0.663614943097232, 0.549961018854458],
                               [-26.644998295726008, -28.533472658505289, -21.514323077327184]])
    elif heading == 'NW':
        corr_param = np.array([[-0.008755509253829, -0.007214571245483, -0.008440157664704],
                               [1.056811056809333, 0.889258681223228, 0.918765120634289],
                               [-29.514419565085220, -32.379106503389259, -26.934729925348211]])
    depr_angle_max = 40  # Maximum depression angle for P-band

# Compute the correction map in dB
correction_map_dB = corr_param[0, pol_ind] * (depression_angle * 180 / np.pi)**2 + \
                    corr_param[1, pol_ind] * (depression_angle * 180 / np.pi) + \
                    corr_param[2, pol_ind]

# Compute the maximum correction in dB for angles greater than maximum depression angle
correction_max_dB = corr_param[0, pol_ind] * depr_angle_max**2 + \
                    corr_param[1, pol_ind] * depr_angle_max + \
                    corr_param[2, pol_ind]

# Apply maximum correction for angles beyond the maximum depression angle
correction_map_dB[depression_angle * 180 / np.pi > depr_angle_max] = correction_max_dB
correction_map_dB = correction_map_dB - correction_max_dB  # Normalize the correction map

# Define canopy height range
z_min = 10  # Minimum height [m]
z_max = 40  # Maximum height [m]

# Find indices corresponding to the defined height range
z_min_ind = np.argmin(np.abs(z_axis - z_min))
z_max_ind = np.argmin(np.abs(z_axis - z_max))

# Compute mean power within the defined height range
Pcanopy = np.mean(Pcube[:, :, z_min_ind:z_max_ind+1], axis=2)

# Convert power to dB
Pcanopy_dB = 10 * np.log10(Pcanopy)

# Apply the near range ramp correction
Pcanopy_de_ramped_dB = Pcanopy_dB - correction_map_dB

# Plotting the TomoSAR intensity before correction
plt.figure()
plt.imshow(Pcanopy_dB, extent=[azimuth_axis[0]*1e-3, azimuth_axis[-1]*1e-3, ground_range_axis[0]*1e-3, ground_range_axis[-1]*1e-3], vmin=-20, vmax=10)
plt.xlabel('Azimuth [km]')
plt.ylabel('Ground range [km]')
plt.title('TomoSAR intensity')
plt.colorbar()
plt.show()

# Plotting the TomoSAR intensity after correction
plt.figure()
plt.imshow(Pcanopy_de_ramped_dB, extent=[azimuth_axis[0]*1e-3, azimuth_axis[-1]*1e-3, ground_range_axis[0]*1e-3, ground_range_axis[-1]*1e-3], vmin=-20, vmax=10)
plt.xlabel('Azimuth [km]')
plt.ylabel('Ground range [km]')
plt.title('TomoSAR intensity deramped')
plt.colorbar()
plt.show()

# Loading maps for geocoding "Pcube" data
with Dataset(MetaSensing_slc_fullpath, 'r') as ds:
    LatImage = ds.variables['LatImage'][gr_start_ind:gr_stop_ind+1, az_start_ind:az_stop_ind+1]
    LonImage = ds.variables['LonImage'][gr_start_ind:gr_stop_ind+1, az_start_ind:az_stop_ind+1]


In [5]:
# test on local data 
# Options
parent_path = '/Users/gracecolverd/fdl_24'

# Either 'P' or 'L'
band_opt = 'P'

# Either 'NW' (290°) or 'SE' (110°)
heading = 'NW'

# Either 'Monostatic' or 'Bistatic' (available for L-band only)
acquisition_mode = 'Monostatic'

# 'HH', 'HV' or 'VV'
polarimetric_channel = 'HH'


if acquisition_mode == 'Monostatic':
    mono_bi_ind = 1
    mono_bi_str = 'mono'
elif acquisition_mode == 'Bistatic':
    mono_bi_ind = 2
    mono_bi_str = 'bi'

# Determine the polarimetric channel index
if polarimetric_channel == 'HH':
    pol_ind = 1
elif polarimetric_channel == 'HV':
    pol_ind = 2
elif polarimetric_channel == 'VV':
    pol_ind = 3

# Load local coordinates for ground range and azimuth axis
ground_range_axis = loadmat('/Users/gracecolverd/fdl_24/UTIL/datacube/ground_range_axis.mat')['ground_range_axis'].squeeze()
azimuth_axis = loadmat('/Users/gracecolverd/fdl_24/UTIL/datacube//azimuth_axis.mat')['azimuth_axis'].squeeze()
Ngr = len(ground_range_axis)  # Number of ground range points
Naz = len(azimuth_axis)       # Number of azimuth points

# Determine the MetaSensing SLC full path based on band and heading options
if band_opt == 'P':
    if heading == 'NW':
        MetaSensing_slc_fullpath = '/Users/gracecolverd/fdl_24/MetaSensing/SAR_CPLX_20200722092915_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc_.gstmp'
  

# Read the MetaSensing SLC data using netCDF4
with Dataset(MetaSensing_slc_fullpath, 'r') as ds:
    MetaSensing_slc_azimuth_axis = ds.variables['CrossRange'][:]
    MetaSensing_slc_ground_range_axis = ds.variables['GroundRange'][:]
    DTM = ds.variables['DEMImage'][:]

# Find the indices for azimuth and ground range based on the loaded axes
az_start_ind = np.argmin(np.abs(MetaSensing_slc_azimuth_axis - azimuth_axis[0]))
az_stop_ind = np.argmin(np.abs(MetaSensing_slc_azimuth_axis - azimuth_axis[-1]))
gr_start_ind = np.argmin(np.abs(MetaSensing_slc_ground_range_axis - ground_range_axis[0]))
gr_stop_ind = np.argmin(np.abs(MetaSensing_slc_ground_range_axis - ground_range_axis[-1]))

# Slice the DTM to match the indices found
DTM = DTM[gr_start_ind:gr_stop_ind+1, az_start_ind:az_stop_ind+1]

# Define the z-axis range and initialize the data cubes
z_axis = np.arange(-10, 62, 2)
Nz = len(z_axis)  # Number of z-axis points
Pcube = np.zeros((Ngr, Naz, Nz))  # Initialize power cube
PcubeNorm = np.zeros((Ngr, Naz, Nz))  # Initialize normalized power cube (tomographic coherence)

# # Function to read biomass data (assumed to be implemented)
# def read_biomass_data(filepath, param, mode):
#     # Placeholder function
#     # You need to implement this based on the actual data format
#     pass

# Loop through each z-axis point and load the corresponding data
for z_ind in range(Nz):
    print(f'Loading {z_axis[z_ind]}m ({z_ind+1}/{Nz})...')
    
    # Determine the file paths for the current z-slice
    if band_opt == 'P':
        curr_fullpath = f'/Users/gracecolverd/fdl_24/DataCube/ComplexSlice/tomo_slice_{z_axis[z_ind]}m_{polarimetric_channel}.dat'
        curr_fullpathP = f'/Users/gracecolverd/fdl_24/DataCube/Power/Pbp_{z_axis[z_ind]}m_{polarimetric_channel}.dat'

    print(curr_fullpath)
    # Load the data and normalize it
    Pcube[:, :, z_ind] = np.abs(read_biomass_data(curr_fullpath, [], 1))**2
    temp = read_biomass_data(curr_fullpathP, [], 2)
    PcubeNorm[:, :, z_ind] = Pcube[:, :, z_ind] / (temp + np.finfo(float).eps)  # Normalize to avoid division by zero
    print('done.')

# Plot the normalized power cube for a specific ground range slice
plt.figure()
plt.imshow(np.transpose(PcubeNorm[Ngr//3, :, :], (1, 0)), aspect='auto', extent=[azimuth_axis[0]*1e-3, azimuth_axis[-1]*1e-3, z_axis[-1], z_axis[0]])
plt.axis('xy')
plt.xlabel('Azimuth [km]')
plt.ylabel('[m]')
plt.show()



Loading -10m (1/36)...
/Users/gracecolverd/fdl_24/DataCube/ComplexSlice/tomo_slice_-10m_HH.dat
Error in read_biomass_data: File cannot be opened.


TypeError: bad operand type for abs(): 'NoneType'

In [5]:
Ngr

1274